# Очистка таблиц Users, Items и Users_Items

В этом ноутбуке я привожу таблицы в нормальный вид, оставляю только нужные столбцы и строки.

## Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import os
import scipy.stats as sts
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from pathlib import Path

## Загрузка данных

In [2]:
BASE_PATH = os.path.abspath("../../../Tables/BaseTable")

In [3]:
users_df = pd.read_csv(os.path.join(BASE_PATH, 'Users.csv'), sep=';', encoding='cp1251')
items_df = pd.read_csv(os.path.join(BASE_PATH, 'Items.csv'), sep=';', encoding='cp1251')
user_item_df = pd.read_csv(os.path.join(BASE_PATH, 'User_Items_test_month.csv'), sep=';', encoding='cp1251')

In [4]:
print("Изначальные размеры:")
print(f"Users: {users_df.shape}")
print(f"Items: {items_df.shape}")
print(f"User_Items: {user_item_df.shape}")

Изначальные размеры:
Users: (2893, 18)
Items: (258, 8)
User_Items: (3133, 17)


In [14]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 18 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Имя                                        2892 non-null   str    
 1   Телефон                                    2892 non-null   str    
 2   Email                                      997 non-null    str    
 3   Категории                                  58 non-null     str    
 4   Дата рождения                              2 non-null      str    
 5   Потратил, ?                                2892 non-null   str    
 6   Оплатил, ?                                 2892 non-null   str    
 7   Пол                                        6 non-null      str    
 8   Карта                                      0 non-null      float64
 9   Скидка                                     2892 non-null   float64
 10  Последний визит                    

In [15]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


In [16]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    1777 non-null   str    
 1   Специализация мастера           1777 non-null   str    
 2   Имя клиента                     1154 non-null   str    
 3   	Телефон                        1154 non-null   float64
 4   Email                           502 non-null    str    
 5   Время визита                    1777 non-null   str    
 6   Имя	 создателя                  1777 non-null   str    
 7   Дата создания                   1777 non-null   str    
 8   Статус                          1777 non-null   str    
 9   Источник                        1777 non-null   str    
 10  Категория услуги                3096 non-null   str    
 11  Название услуги                 3096 non-null   str    
 12  Стоимость, руб                  3096 non-null

Заранее изучив данные, выявлены необходимые операции:

__Для Users__

1) Вместо Имя+Телефон+Email ввести `id_user`, а Email удалить
2) Удалить столбцы Дата рождения, Потратил в руб; Карта; Скидка; Последний визит; Чаевые; Количество посещений; Комментарий; Дополнительный телефон; Согласен на получение рассылок; Согласен на обработку персональных данных т.к. они большинством своим не информативны либо полностью пустые
3) В столбце Категория есть несколько возможных значений я хочу:
- все пропуски и значения "Клиенты отщепенцев", "ПРЕДОПЛАТА" заменить на "обычный"
- удалить все экземпляры (строчки), где категория "Черный список"
- заменить значение "Постоянный" на "обычный" (т.к. в таблице только 1 такой экземпляр)
4) Удалить пропуски, если они останутся

__Для Items__

1) Удалить все услуги, где значение "Категория товара или услуги в продаже" = "Солярий" (слишком много посещений и направление никак не связано с остальными услугами)
2) Удалить все услуги, которые были оказаны слишком маленькое количество раз ("Количество оказанных услуг"<8 раз) или где количество уникальных клиентов слишком маленькое ("Уникальные клиенты" < 7).
3) По сути каждая строчка это и есть информация об уникальной услуге, но я бы добавил столбец `id_item` и сделал его PK

__Для User-Items__

1) Удалить все элементы связанные с солярием
2) Заменить имена столбцов на отношение сущьностей (то есть столбец "Имя", который относиться к клиенту назвать "Имя клиента" и т.п.)
3) Удалить Создал (Имя, Дата);Статус; Источник; Информация (Статус, Имя, Дата); Оплачено полностью; Комментарий; Категория (они не имеют смысла или нулевые)
4) Добавить `user_id` (для каждого клиента такой же как в таблице Users(то есть чтобы одному клиенту и там и там значился 1 и тот же id)), добавить `item_id` (по аналогии с `user_id`) и удалить "Email"

## Обработка Users

#### Вводим user_id для каждого уникального пользователя, определяя его с помощью тройки Имя+телефон+Email

In [17]:
users_df['id_user'] = pd.Categorical(users_df['Имя'].astype(str) + '_' +
                                   users_df['Телефон'].astype(str) + '_' +
                                   users_df['Email'].astype(str)).codes

#### Настройка столбца 'Категории'

In [18]:
users_df['Категории'] = users_df['Категории'].fillna('обычный')
users_df['Категории'] = users_df['Категории'].replace({
    'Клиенты отщепенцев': 'обычный',
    'ПРЕДОПЛАТА': 'обычный',
    'Постоянный': 'обычный'
})

In [19]:
users_df['Категории'].value_counts()

Категории
обычный          2881
Черный список      12
Name: count, dtype: int64

In [20]:
users_df = users_df[users_df['Категории'] != 'Черный список'].copy()
users_df['Категории'].value_counts()

Категории
обычный    2881
Name: count, dtype: int64

Оставляем столбец, т.к. в будущем возможно введение других категорий.

In [21]:
users_df = users_df[[
    'id_user', 'Имя', 'Телефон',
    'Категории', 'Оплатил, ?', 'Последний визит'
]].copy()
users_df = users_df.rename(columns={'Оплатил, ?': 'Оплачено в руб'})

In [22]:
users_df.info()

<class 'pandas.DataFrame'>
Index: 2881 entries, 0 to 2892
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2881 non-null   int16
 1   Имя              2880 non-null   str  
 2   Телефон          2880 non-null   str  
 3   Категории        2881 non-null   str  
 4   Оплачено в руб   2880 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 140.7 KB


#### Удалим пропуски и зафиксируем таблицу

In [23]:
users_clean = users_df.dropna(subset=['Последний визит']).copy()

In [24]:
users_clean.isna().sum()

id_user            0
Имя                0
Телефон            0
Категории          0
Оплачено в руб     0
Последний визит    0
dtype: int64

In [25]:
users_clean.info()

<class 'pandas.DataFrame'>
Index: 2711 entries, 0 to 2719
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2711 non-null   int16
 1   Имя              2711 non-null   str  
 2   Телефон          2711 non-null   str  
 3   Категории        2711 non-null   str  
 4   Оплачено в руб   2711 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 132.4 KB


#### Настроем типы данных

In [26]:
users_clean['Оплачено в руб'] = users_clean['Оплачено в руб'].astype(str).str.replace(',', '.').astype(float)

In [27]:
users_clean['Последний визит'] = pd.to_datetime(users_clean['Последний визит'], errors='coerce')

In [28]:
users_clean.dtypes

id_user                     int16
Имя                           str
Телефон                       str
Категории                     str
Оплачено в руб            float64
Последний визит    datetime64[us]
dtype: object

## Обработка Items

In [29]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


#### Добавим items_id

In [30]:
items_df['id_item'] = range(len(items_df))

#### Удалим все услуги связанные с солярием
(он не связан с остальным бизнессом и имеет оторванную от других данных размерность, он не нужен для наших задач)

In [31]:
items_df = items_df[
    items_df['Категория товара или услуги в продаже'] != 'Солярий'
].copy()

#### Удалим лишние услуги
(Это услуги, которые не пользовались спросом в последний год)

In [32]:
items_clean = items_df[
    (items_df['Количество оказанных услуг'] >= 8) &
    (items_df['Уникальные клиенты'] >= 7)
].copy()

In [33]:
items_clean.info()

<class 'pandas.DataFrame'>
Index: 136 entries, 1 to 257
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             136 non-null    str  
 1   Категория товара или услуги в продаже  136 non-null    str  
 2   Количество оказанных услуг             136 non-null    int64
 3   Доля от оказанных услуг в %            136 non-null    str  
 4   Выручка от продажи услуг, ?            136 non-null    str  
 5   % от общей выручки за услуги           136 non-null    str  
 6   Ср. выручка с одного клиента, ?        136 non-null    str  
 7   Уникальные клиенты                     136 non-null    int64
 8   id_item                                136 non-null    int64
dtypes: int64(3), str(6)
memory usage: 10.6 KB


In [34]:
items_clean = items_clean.rename(columns={'Выручка от продажи услуг, ?': 'Выручка от продажи услуг, руб'})
items_clean = items_clean.rename(columns={'Ср. выручка с одного клиента, ?': 'Ср. выручка с одного клиента, руб'})

#### Настраиваем типы данных

In [35]:
numeric_cols = [
    'Количество оказанных услуг',
    'Доля от оказанных услуг в %',
    'Выручка от продажи услуг, руб',
    'Ср. выручка с одного клиента, руб',
    '% от общей выручки за услуги',
    'Уникальные клиенты'
]

for col in numeric_cols:
  items_clean[col] = items_clean[col].astype(str).str.replace(',', '.').replace(' ', '').astype(float)

In [36]:
items_clean.info()

<class 'pandas.DataFrame'>
Index: 136 entries, 1 to 257
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Название услуги или товара             136 non-null    str    
 1   Категория товара или услуги в продаже  136 non-null    str    
 2   Количество оказанных услуг             136 non-null    float64
 3   Доля от оказанных услуг в %            136 non-null    float64
 4   Выручка от продажи услуг, руб          136 non-null    float64
 5   % от общей выручки за услуги           136 non-null    float64
 6   Ср. выручка с одного клиента, руб      136 non-null    float64
 7   Уникальные клиенты                     136 non-null    float64
 8   id_item                                136 non-null    int64  
dtypes: float64(6), int64(1), str(2)
memory usage: 10.6 KB


## Обработка User-Items

In [5]:
user_item_df = user_item_df.rename(columns={'Имя   создателя': 'Имя создателя'})
user_item_df = user_item_df.rename(columns={'Имя	 мастера ': 'Имя мастера'})
user_item_df = user_item_df.rename(columns={'	Телефон ': 'Телефон'})

In [6]:
user_item_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     1777 non-null   object 
 1   Специализация мастера           1777 non-null   object 
 2   Имя клиента                     1154 non-null   object 
 3   Телефон                         1154 non-null   float64
 4   Email                           502 non-null    object 
 5   Время визита                    1777 non-null   object 
 6   Имя	 создателя                  1777 non-null   object 
 7   Дата создания                   1777 non-null   object 
 8   Статус                          1777 non-null   object 
 9   Источник                        1777 non-null   object 
 10  Категория услуги                3096 non-null   object 
 11  Название услуги                 3096 non-null   object 
 12  Стоимость, руб                  30

#### Удаляем все записи с солярием

In [7]:
user_item_df = user_item_df[(user_item_df['Специализация мастера '] != 'Солярий') &
                            ((user_item_df['Специализация мастера '] != 'Администратор'))].copy()

In [8]:
user_item_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2465 entries, 4 to 3131
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     1109 non-null   object 
 1   Специализация мастера           1109 non-null   object 
 2   Имя клиента                     1100 non-null   object 
 3   Телефон                         1100 non-null   float64
 4   Email                           487 non-null    object 
 5   Время визита                    1109 non-null   object 
 6   Имя	 создателя                  1109 non-null   object 
 7   Дата создания                   1109 non-null   object 
 8   Статус                          1109 non-null   object 
 9   Источник                        1109 non-null   object 
 10  Категория услуги                2464 non-null   object 
 11  Название услуги                 2464 non-null   object 
 12  Стоимость, руб                  2464 no

Удаляем неинформативные столбцы

In [10]:
keep_cols = [
    'Имя клиента ', 'Телефон', 'Email ',           # Клиент
    'Имя мастера', 'Специализация мастера ',     # Мастер
    'Время визита',                             # Время
    'Категория услуги', 'Название услуги ',      # Услуги
    'Стоимость с учетом скидки, руб'            # Weight
]
user_items_clean = user_item_df[keep_cols].copy()

#### Заполнение пропусков 

In [11]:
key_cols = ['Имя клиента ', 'Телефон']
service_col = 'Название услуги '

Сортируем по паре телефон + время визита 

In [13]:
user_items_clean = user_items_clean.sort_values(['Телефон', 'Время визита']).copy()

Заполняем пропуски значениями из предыдущей строки

In [15]:
fill_cols = [col for col in user_items_clean.columns if col != service_col]
user_items_clean[fill_cols] = user_items_clean[fill_cols].fillna(method='ffill')

In [16]:
user_items_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2465 entries, 804 to 3131
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя клиента                     2465 non-null   object 
 1   Телефон                         2465 non-null   float64
 2   Email                           2463 non-null   object 
 3   Имя мастера                     2465 non-null   object 
 4   Специализация мастера           2465 non-null   object 
 5   Время визита                    2465 non-null   object 
 6   Категория услуги                2465 non-null   object 
 7   Название услуги                 2464 non-null   object 
 8   Стоимость с учетом скидки, руб  2465 non-null   object 
dtypes: float64(1), object(8)
memory usage: 192.6+ KB


#### Удаляем строки с пропусками данных 

In [17]:
user_items_clean = user_items_clean[
    (user_items_clean[service_col].notna()) & 
    (user_items_clean[service_col].str.strip() != '') &
    (user_items_clean['Имя клиента '].notna())
].copy()

In [22]:
print(f"Количество уникальных клиентов: {len(user_items_clean['Телефон'].unique())}")

Количество уникальных клиентов: 749


#### Подстановка `user_id` и `item_id` из `users_clean` и `items_clean`
Для этого подправим запись телеона во всех таблицах и проведем маппинг.  


In [42]:
print(f"\nТип Телефон в user_items: {user_items_clean['Телефон'].dtype}")
print(f"Тип Телефон в users_clean: {users_clean['Телефон'].dtype}")
print(f"Примеры Телефон user_items: {user_items_clean['Телефон'].head().tolist()}")


Тип Телефон в user_items: float64
Тип Телефон в users_clean: str
Примеры Телефон user_items: [79091590899.0, nan, nan, 79268226086.0, 79252466666.0]


In [43]:
key_cols = ['Имя клиента ', 'Телефон', 'Название услуги ']
user_items_clean = user_items_clean.dropna(subset=key_cols).copy()
user_items_clean.isna().sum()

Имя клиента                         0
Телефон                             0
Email                             612
Имя мастера                         0
Специализация мастера               0
Время визита                        0
Категория услуги                    0
Название услуги                     0
Стоимость с учетом скидки, руб      0
dtype: int64

In [44]:
def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone_str = str(phone).replace('.0', '').replace('+7', '8')
    return phone_str.strip()

users_clean['Телефон'] = users_clean['Телефон'].apply(clean_phone)
user_items_clean['Телефон'] = user_items_clean['Телефон'].apply(clean_phone)

In [45]:
user_map = users_clean.set_index(['Имя', 'Телефон'])['id_user'].to_dict()

user_items_clean['user_id'] = user_items_clean.apply(
    lambda x: user_map.get((x['Имя клиента '], x['Телефон'])), axis=1
)

In [46]:
print(f"\n✅ user_id заполнено: {user_items_clean['user_id'].notna().sum()}/{len(user_items_clean)}")


✅ user_id заполнено: 1093/1099


In [47]:
user_items_clean.isna().sum()

Имя клиента                         0
Телефон                             0
Email                             612
Имя мастера                         0
Специализация мастера               0
Время визита                        0
Категория услуги                    0
Название услуги                     0
Стоимость с учетом скидки, руб      0
user_id                             6
dtype: int64

In [48]:
item_map = dict(zip(items_clean['Название услуги или товара'], items_clean['id_item']))

user_items_clean['item_id'] = user_items_clean['Название услуги '].map(item_map)

In [49]:
user_items_clean.isna().sum()

Имя клиента                         0
Телефон                             0
Email                             612
Имя мастера                         0
Специализация мастера               0
Время визита                        0
Категория услуги                    0
Название услуги                     0
Стоимость с учетом скидки, руб      0
user_id                             6
item_id                            22
dtype: int64

In [50]:
user_items_final = user_items_clean.drop(columns=['Email ']).copy()
user_items_final = user_items_final.dropna(subset=['user_id', 'item_id'])

In [51]:
user_items_final.info()

<class 'pandas.DataFrame'>
Index: 1072 entries, 8 to 3129
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя клиента                     1072 non-null   str    
 1   Телефон                         1072 non-null   str    
 2   Имя мастера                     1072 non-null   str    
 3   Специализация мастера           1072 non-null   str    
 4   Время визита                    1072 non-null   str    
 5   Категория услуги                1072 non-null   str    
 6   Название услуги                 1072 non-null   str    
 7   Стоимость с учетом скидки, руб  1072 non-null   str    
 8   user_id                         1072 non-null   float64
 9   item_id                         1072 non-null   float64
dtypes: float64(2), str(8)
memory usage: 92.1 KB


#### Обработка типов данных

In [55]:
# 1. user_id и item_id → int32 (убираем float)
user_items_final['user_id'] = user_items_final['user_id'].astype('int32')
user_items_final['item_id'] = user_items_final['item_id'].astype('int32')

# 2. Стоимость с учетом скидки → float64
user_items_final['Стоимость с учетом скидки, руб'] = (
    user_items_final['Стоимость с учетом скидки, руб']
    .astype(str)
    .str.replace(',', '.')
    .str.replace(' ', '')
    .astype(float)
)

#Время визита → datetime
user_items_final['Время визита'] = pd.to_datetime(
    user_items_final['Время визита'],
    errors='coerce',
    dayfirst=True
)

# 3. Проверяем результат
print("✅ Типы после конвертации:")
print(user_items_final.dtypes)
print(f"\nРазмер: {user_items_final.shape}")


✅ Типы после конвертации:
Имя клиента                                  str
Телефон                                      str
Имя мастера                                  str
Специализация мастера                        str
Время визита                      datetime64[us]
Категория услуги                             str
Название услуги                              str
Стоимость с учетом скидки, руб           float64
user_id                                    int32
item_id                                    int32
dtype: object

Размер: (1072, 10)


## Сортировка таблиц по id

In [53]:
# 1. users_clean по user_id
users_clean = users_clean.sort_values('id_user').reset_index(drop=True)

# 2. items_clean по item_id
items_clean = items_clean.sort_values('id_item').reset_index(drop=True)

# 3. user_items_final по user_id
user_item_final = user_items_final.sort_values('user_id').reset_index(drop=True)

## Выгрузка таблиц

In [56]:
''' 
users_clean.to_csv('users_clean_final.csv', index=False, encoding='utf-8-sig')
items_clean.to_csv('items_clean_final.csv', index=False, encoding='utf-8-sig')
user_item_final.to_csv('user_items_final.csv', index=False, encoding='utf-8-sig')
'''

" \nusers_clean.to_csv('users_clean_final.csv', index=False, encoding='utf-8-sig')\nitems_clean.to_csv('items_clean_final.csv', index=False, encoding='utf-8-sig')\nuser_item_final.to_csv('user_items_final.csv', index=False, encoding='utf-8-sig')\n"